# Finetune BERT for Downstream Tasks

*How much does BERT improve sentiment classification, why, and under what conditions?*

Description:
* Task: Binary sentiment classification (positive / negative)
* Dataset: IMDb (50k reviews with label, 50k without label)

### Load libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# torch stuff
import torch

from torch.utils.data import DataLoader

# dataset
from datasets import load_dataset
dataset = load_dataset('imdb')

# model
from transformers import BertTokenizer, BertForSequenceClassification, AutoModelForSequenceClassification

/Users/emilfuhr/Desktop/Uni/Data Science/2. Semester/Advanced Machine Learning/Project/Mini-Project-AML-2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 733403.28 examples/s]


# Dataset

Each instance consists of a text field and label (0 or 1)


`unsupervised`, label = -1

### Data preprocessing 

In [ ]:
# split training data into 90/10 train/validation split
dataset_train, dataset_eval = dataset['train'].train_test_split(test_size=0.1, seed=42)

In [25]:
# show an instance of the dataset
print(dataset['train'][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [26]:
print(dataset['unsupervised'][0])

{'text': 'This is just a precious little diamond. The play, the script are excellent. I cant compare this movie with anything else, maybe except the movie "Leon" wonderfully played by Jean Reno and Natalie Portman. But... What can I say about this one? This is the best movie Anne Parillaud has ever played in (See please "Frankie Starlight", she\'s speaking English there) to see what I mean. The story of young punk girl Nikita, taken into the depraved world of the secret government forces has been exceptionally over used by Americans. Never mind the "Point of no return" and especially the "La femme Nikita" TV series. They cannot compare the original believe me! Trash these videos. Buy this one, do not rent it, BUY it. BTW beware of the subtitles of the LA company which "translate" the US release. What a disgrace! If you cant understand French, get a dubbed version. But you\'ll regret later :)', 'label': -1}


In [27]:
dataset.column_names

{'train': ['text', 'label'],
 'test': ['text', 'label'],
 'unsupervised': ['text', 'label']}

In [28]:
print ('train:', len(dataset['train']))
print ('unsupervised:', len(dataset['unsupervised']))
print ('test:', len(dataset['test']))


train: 25000
unsupervised: 50000
test: 25000


### Split into batches

#### Tokenization

`text` input has to be processed and tokenized

In [29]:
tokenizer = BertTokenizer.from_pretrained('bert-base-cased') # or "uncased"

In [ ]:
encoded_dataset_train = dataset_train.map(lambda x: tokenizer(x['text'], truncation=True, max_length=256), batched=True) 
encoded_dataset_eval = dataset_eval.map(lambda x: tokenizer(x['text'], truncation=True, max_length=256), batched=True) 

Map: 100%|██████████| 50000/50000 [00:05<00:00, 8404.07 examples/s]


In [ ]:
encoded_dataset_train.column_names

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
 'unsupervised': ['text',
  'label',
  'input_ids',
  'token_type_ids',
  'attention_mask']}

In [ ]:
print (len(encoded_dataset_train['train'][0]['text']))

1640


In [ ]:
print (encoded_dataset_train['train'][0]['input_ids'])
print (len(encoded_dataset_train['train'][0]['input_ids']))

[101, 146, 12765, 146, 6586, 140, 19556, 19368, 13329, 118, 162, 21678, 2162, 17056, 1121, 1139, 1888, 2984, 1272, 1104, 1155, 1103, 6392, 1115, 4405, 1122, 1165, 1122, 1108, 1148, 1308, 1107, 2573, 119, 146, 1145, 1767, 1115, 1120, 1148, 1122, 1108, 7842, 1118, 158, 119, 156, 119, 10148, 1191, 1122, 1518, 1793, 1106, 3873, 1142, 1583, 117, 3335, 1217, 170, 5442, 1104, 2441, 1737, 107, 6241, 107, 146, 1541, 1125, 1106, 1267, 1142, 1111, 1991, 119, 133, 9304, 120, 135, 133, 9304, 120, 135, 1109, 4928, 1110, 8663, 1213, 170, 1685, 3619, 3362, 2377, 1417, 14960, 1150, 3349, 1106, 3858, 1917, 1131, 1169, 1164, 1297, 119, 1130, 2440, 1131, 3349, 1106, 2817, 1123, 2209, 1116, 1106, 1543, 1199, 3271, 1104, 4148, 1113, 1184, 1103, 1903, 156, 11547, 1162, 1354, 1164, 2218, 1741, 2492, 1216, 1112, 1103, 4357, 1414, 1105, 1886, 2492, 1107, 1103, 1244, 1311, 119, 1130, 1206, 4107, 8673, 1105, 6655, 10552, 3708, 2316, 1104, 8583, 1164, 1147, 11089, 1113, 4039, 117, 1131, 1144, 2673, 1114, 1123, 336

In [ ]:
print (encoded_dataset_train['train'][0]['token_type_ids'])
print (len(encoded_dataset_train['train'][0]['token_type_ids']))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
256


In [ ]:
print(encoded_dataset_train['train'][0]['attention_mask'])
print(len(encoded_dataset_train['train'][0]['attention_mask']))

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
256


# Model

## Baseline:
Suggested baseline: TF-IDF + Logistic Regression $\to$ Show a simple, interpretable reference point

Then report
- Accuracy, loss, F1-score? ...
- Training time (optional but nice)

In [6]:
# Initialize a BERT model for binary classification
model_name = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print(model.config)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6645.96it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



## Weights & Biases

In [4]:
import wandb

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/emilfuhr/.netrc.
wandb: Currently logged in as: emilfuhr (emilfuhr-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb_run = wandb.init(
    project="mini-project-aml-2026",
    entity="Mini-Project-AML-2026",
    name="bert-base-cased",
    config={
        "model_name": model_name,
        "num_labels": 2,
        "max_length": 256,
    }
)

In [8]:
def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()
print(f"Using device: {device}")

Using device: cpu


## Bert (Fine-tuning)

**Core components**
- Pretrained model (e.g. bert-base-XX)
- Tokenization (module from HuggingFace)
- Loss: XX

In [38]:

def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if torch.cuda.device_count() >= i + 1:
      return torch.device(f'cuda:{i}')
    return torch.device('cpu')
device = try_gpu()

print(f"Using device: {device}")

Using device: cpu


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
        num_train_epochs=3,              # Number of epochs
        per_device_train_batch_size=16,  # Batch size per GPU
        per_device_eval_batch_size=16,
        learning_rate=5e-5,              # Start with a small learning rate
        # label_smoothing_factor=0.0,
        weight_decay=0.1,
        # max_grad_norm=1.0,
        save_total_limit=2,              # Limit checkpoints to save space

        # Transformers-specific config
        logging_steps=100,
        load_best_model_at_end=True, # Trainer will load the checkpoint with the best eval metric at the end of training 
        eval_strategy="epoch",
        save_strategy="epoch",
        # metric_for_best_model="accuracy",  # TODO: Do they pick the model with least validation accuracy or the final model?
        # report_to=[],  # Disable logging integrations by default
        # seed=42,
        # fp16=True,                        # Enable mixed precision for faster training

        dataloader_pin_memory=True if torch.cuda.is_available() else False,
        output_dir=str("./results"),
        report_to = "wandb",          # tells the Trainer to push logs to W&B
        run_name = "bert-base-cased", # shown in the W&B UI
)
# # Define training arguments
# training_args = TrainingArguments(
#     output_dir="./results",           # Directory for saving model checkpoints
#     evaluation_strategy="epoch",     # Evaluate at the end of each epoch
#     learning_rate=5e-5,              # Start with a small learning rate
#     per_device_train_batch_size=16,  # Batch size per GPU
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,              # Number of epochs
#     weight_decay=0.01,               # Regularization
#     save_total_limit=2,              # Limit checkpoints to save space
#     load_best_model_at_end=True,     # Automatically load the best checkpoint
#     logging_dir="./logs",            # Directory for logs
#     logging_steps=100,               # Log every 100 steps
#     fp16=True                        # Enable mixed precision for faster training
# )

# print(args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=False,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fal

In [40]:
from transformers import Trainer
from evaluate import load

# Load a metric (F1-score in this case)
metric = load("f1")

# Define a custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [41]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
trainer = Trainer(
    model=model,                        # Pre-trained BERT model
    args=args,                 # Training arguments
    train_dataset=encoded_dataset_train,
    eval_dataset=encoded_dataset_eval,
    # tokenizer=tokenizer,
    data_collator=data_collator,        # Efficient batching
    compute_metrics=compute_metrics     # Custom metric
)

# Start training
trainer.train()

# finish w&b logging
wandb.finish()   # cleanly closes the run once training is done

Epoch,Training Loss,Validation Loss,F1
1,0.268919,0.225664,0.912107
2,0.145939,0.281419,0.913904
3,0.068733,0.383561,0.916189


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.08it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=4689, training_loss=0.1784705496413999, metrics={'train_runtime': 3468.162, 'train_samples_per_second': 21.625, 'train_steps_per_second': 1.352, 'total_flos': 9866639909338560.0, 'train_loss': 0.1784705496413999, 'epoch': 3.0})

In [43]:
# Evaluate the model
results = trainer.evaluate()
print(results)

RuntimeError: on_train_begin must be called before on_evaluate

### Sweep wandb configuration

In [ ]:
sweep_config = {
    "method": "bayes",  # or "random" or "grid"
    "metric": {
        "name": "eval/f1",
        "goal": "maximize"
    },
    "parameters": {
        "model_name": {
            "values": ["bert-base-cased", "bert-base-uncased", "roberta-base"]
        },
        "learning_rate": {
            "values": [2e-5, 5e-5, 1e-4]
        },
        "num_train_epochs": {
            "values": [2, 3, 5]
        },
        "weight_decay": {
            "values": [0.0, 0.1]
        },
    }
}

sweep_id = wandb.sweep(sweep_config, project="mini-project-aml-2026")

In [ ]:
def train():
    wandb.init()
    cfg = wandb.config  # access config params for this run

    # Load model using sweep param
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name, num_labels=2
    )

    args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=cfg.num_train_epochs,
        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_steps=100,
        report_to="wandb",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    wandb.finish()

# Launch the sweep
wandb.agent(sweep_id, function=train, count=10)  # count --> max runs

### Hyperparameter tuning

## Evaluation

* Compare baseline with Bert model 
* Assumption: Bert improves sentiment analysis...